# NCM / kNN diagnostic probes — C1, C2, C3 (§7 v5.2)

Alternative fixed-encoder classifiers on top of the SAME cached
features the linear probe already used, val-only, seed 42, no test
contact. Motivation (§7): the linear probe is one specific classifier
choice; NCM/kNN answer "is the activity structure still there under a
different readout", and kNN in particular is the standard evaluation
metric for contrastive (SupCon) representations in the literature —
if C3's linear-probe number understates its representation quality,
kNN is where that would show up.

**Thin notebook (promotion 2026-07-18):** the NCM/kNN scorers live in
`sharp_har/diagnostics.py` — they are pre-registered §7 metrics with
doc-declared hyperparameters, re-run across sessions, so they are
package code per the thin-notebook rule (cross-review recorded in
STATUS; math verified byte-identical to the notebook-local cell that
produced the first C1 numbers). The frozen §5.3 linear-probe recipe
in `probe.py` stays untouched; `harness.fuse_windows`/`macro_f1` and
`probe.majority_baseline` are reused exactly as everywhere else.

**Declared hyperparameters (§7 v5.2, fixed a priori, identical recipe
for every encoder):**
- features L2-normalized, cosine similarity (on normalized vectors,
  equivalent to Euclidean — standard metric for contrastive features);
- **NCM**: one centroid per class = mean of L2-normalized TRAIN
  features, **re-normalized L2** (without this, centroids of different
  norms would make cosine-argmax and Euclidean-argmin diverge); class
  score = cosine similarity to the centroid;
- **kNN**: k=20 (team call); class score = fraction of votes among the
  k nearest TRAIN neighbors, tie broken by the mean similarity of the
  voting neighbors (implemented as `votes/k + 1e-6 * mean_similarity`
  — the epsilon is far below any real vote-count difference, 1/20, so
  it only ever resolves exact ties);
- antenna fusion = mean of class scores over the 4 antennas, then
  argmax (the same `harness.fuse_windows` "softmax_avg" path used by
  every other stream — it only needs a per-sample class-score array,
  not a true softmax, so it applies unchanged to NCM/kNN scores);
- reference pool (declared, §7 refinement 2026-07-18): centroids and
  neighbors use the single pool of TRAIN (window, antenna) samples
  across all 4 antennas — fusion operates on the query side only.

## Environment setup

Reduced preamble: no data staging, no GPU needed — runs on cached
features already on Drive (train as the reference/fit set, val as the
query set — the exact same caches the linear probes already produced).

In [1]:
from pathlib import Path
import json
import subprocess
import sys

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"

cwd = Path.cwd().resolve()
if (cwd / "sharp_har").exists():
    REPO_DIR = cwd
elif (cwd.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent
elif (cwd.parent.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent.parent
else:
    REPO_DIR = Path("/content/sharp-har")
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

# After the clone, so the file exists on a fresh runtime.
!pip install -q -r {REPO_DIR}/requirements.txt

sys.path.insert(0, str(REPO_DIR))

from google.colab import drive
from sharp_har.utils import read_yaml

drive.mount("/content/drive")  # idempotent
CKPT_ROOT = Path(read_yaml(REPO_DIR / "configs" / "paths.yaml")["ckpt_root"])
print("Repo dir:", REPO_DIR)
print("Checkpoint root:", CKPT_ROOT)

Mounted at /content/drive
Repo dir: /content/sharp-har
Checkpoint root: /content/drive/MyDrive/sharp_har_runs


## Imports

The scorers return a `(n_query, n_classes)` class-score array — the
same shape `harness.fuse_windows` expects from a softmax, so fusion,
metrics and the majority-baseline reference all reuse the frozen
harness/probe code unchanged. All logic is imported from the package
(`sharp_har.diagnostics`, Ref. §7 v5.2) — nothing is defined here.

In [2]:
import numpy as np

from sharp_har.diagnostics import l2norm, ncm_scores, knn_scores
from sharp_har.harness import fuse_windows, macro_f1
from sharp_har.probe import majority_baseline

## Run on C1, C2, C3 (+ the C1_s43 robustness footnote)

C1/C2 use `best.ckpt`'s cache (already produced by the C1-lin/C2-lin
probe sessions). C3 has no `best.ckpt` (§6-C3) — its cache stem is
whatever phase B selected.

**Pre-declared seed-robustness footnote (STATUS 2026-07-18, declared
before any s43 feature cache existed):** NCM/kNN are also run on
`C1_s43`'s cached features — a cache the C1⊕C1′ concat control already
requires, so this costs nothing extra. Declared asymmetry: C1 only.
Its rows are compared to C1's seed-42 rows (there is no s43 linear
probe — declared), and reported as a footnote, not a new row. If the
cache is not on Drive yet the entry is skipped with a note, so this
notebook stays runnable both before and after the caching session.

In [3]:
c3_sel = json.loads((CKPT_ROOT / "C3" / "phase_b_selection.json").read_text())
c3_stem = Path(c3_sel["selected_checkpoint"]).stem

STEMS = {
    "C1": ("C1", "best"),
    "C2": ("C2", "best"),
    "C3": ("C3", c3_stem),
    # Pre-declared seed-robustness footnote (STATUS 2026-07-18): cache
    # produced by the C1(+)C1' concat-control session; skipped with a
    # note if not on Drive yet (declared, not an error).
    "C1_s43": ("C1_s43", "best"),
}

# Reference: the frozen §5.3 linear-probe numbers already on record
# (val macro-F1, fused val accuracy) — printed alongside, never recomputed.
# C1_s43 has no linear probe by design (declared): its rows compare to C1's.
LINEAR_PROBE_REFERENCE = {
    "C1": {"macro_f1": 0.8835, "accuracy": 0.8711},
    "C2": {"macro_f1": 0.8410, "accuracy": 0.8023},
    "C3": {"macro_f1": c3_sel["selected_val_macro_f1"], "accuracy": None},
}

rows = []
skipped = []
for label, (folder, stem) in STEMS.items():
    run_dir = CKPT_ROOT / folder
    train_path = run_dir / f"features_{stem}_train.npz"
    val_path = run_dir / f"features_{stem}_val.npz"
    if not (train_path.exists() and val_path.exists()):
        skipped.append(label)
        print(f"{label}: feature cache not on Drive yet — SKIPPED "
              f"(pre-declared footnote entry, rerun after the caching session)")
        continue
    train = np.load(train_path, allow_pickle=False)
    val = np.load(val_path, allow_pickle=False)

    train_x, train_y = l2norm(train["features"].astype(np.float64)), train["y"]
    val_x, val_y = l2norm(val["features"].astype(np.float64)), val["y"]
    n_classes = len(train["labels"])
    tid, ws = val["trace_id"].astype(object), val["window_start"]

    for clf_name, scores in (
        ("NCM", ncm_scores(train_x, train_y, n_classes, val_x)),
        ("kNN", knn_scores(train_x, train_y, n_classes, val_x)),
    ):
        fused = fuse_windows(scores, val_y, tid, ws)
        acc = float((fused["y_pred"] == fused["y_true"]).mean())
        f1 = macro_f1(fused["y_true"], fused["y_pred"])
        base = majority_baseline(fused["y_true"])
        rows.append({"run": label, "classifier": clf_name, "accuracy": acc,
                     "macro_f1": f1, "majority_baseline": base})
        print(f"{label} [{clf_name}]: acc {acc:.4f}, macro-F1 {f1:.4f} "
              f"(majority baseline {base:.4f})")

print("\n" + "=" * 70)
print(f"{'run':7s} {'classifier':11s} {'accuracy':>9s} {'macroF1':>9s} {'baseline':>9s}")
print("-" * 70)
for r in rows:
    print(f"{r['run']:7s} {r['classifier']:11s} {r['accuracy']:9.4f} "
          f"{r['macro_f1']:9.4f} {r['majority_baseline']:9.4f}")
print("-" * 70)
print("reference (frozen §5.3 linear probe, already on record):")
for label, ref in LINEAR_PROBE_REFERENCE.items():
    acc_s = f"{ref['accuracy']:.4f}" if ref["accuracy"] is not None else "   n/a"
    print(f"{label:7s} {'linear':11s} {acc_s:>9s} {ref['macro_f1']:9.4f}")
if skipped:
    print(f"skipped (cache not on Drive yet): {', '.join(skipped)}")
print("=" * 70)
print("\nReading: if NCM/kNN land close to the linear-probe number, the")
print("activity structure is robust to readout choice. If kNN clearly beats")
print("the linear probe on C3 specifically, the linear recipe was")
print("understating SupCon's representation quality — worth a line in the")
print("report either way. §0.5: differences under ~2 points are 'comparable'.")
print("C1_s43 rows (if present) are the pre-declared seed-robustness")
print("footnote: compare them to C1's rows, seed-42 numbers stay primary.")

C1 [NCM]: acc 0.8653, macro-F1 0.8888 (majority baseline 0.3209)
C1 [kNN]: acc 0.8453, macro-F1 0.8563 (majority baseline 0.3209)
C2 [NCM]: acc 0.7765, macro-F1 0.8176 (majority baseline 0.3209)
C2 [kNN]: acc 0.8424, macro-F1 0.8663 (majority baseline 0.3209)
C3 [NCM]: acc 0.6963, macro-F1 0.7178 (majority baseline 0.3209)
C3 [kNN]: acc 0.7937, macro-F1 0.8047 (majority baseline 0.3209)
C1_s43 [NCM]: acc 0.8567, macro-F1 0.8707 (majority baseline 0.3209)
C1_s43 [kNN]: acc 0.8281, macro-F1 0.8497 (majority baseline 0.3209)

run     classifier   accuracy   macroF1  baseline
----------------------------------------------------------------------
C1      NCM            0.8653    0.8888    0.3209
C1      kNN            0.8453    0.8563    0.3209
C2      NCM            0.7765    0.8176    0.3209
C2      kNN            0.8424    0.8663    0.3209
C3      NCM            0.6963    0.7178    0.3209
C3      kNN            0.7937    0.8047    0.3209
C1_s43  NCM            0.8567    0.8707    0.3209


## Archiving

Download the executed notebook (with outputs) and commit it verbatim
over this file in `notebooks/diagnostics/` (rename to the actual run
date if it differs from the one in the filename), per the folder's
README. Findings that change the plan go in `STATUS.md`.